[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Quantum/Quantum_for_Signal_Processors.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Quantum Computing for Signal Processors

A hype-resistant introduction with a home-field advantage: qubits are [unit vectors](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb), gates are unitary matrices, and the QFT — the algorithm behind quantum's most famous speedups — is *literally the FFT's matrix* ([verified](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) against `np.fft`). Everything simulated exactly in NumPy.

## 1. Pre-requisites

[Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) (unitary matrices, tensor structure helps), [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S7 (the DFT).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# a state of n qubits = a unit vector in C^(2^n); gates = unitaries; measurement = |amplitude|²
def kron_all(mats):
    out = np.array([[1.0+0j]])
    for m in mats: out = np.kron(out, m)
    return out
I2 = np.eye(2); H = np.array([[1, 1], [1, -1]])/np.sqrt(2)
X = np.array([[0, 1], [1, 0]]);
def phase(theta): return np.diag([1, np.exp(1j*theta)])

---
### 🕐 Session 1 of 3 — *Qubits Are Vectors, Gates Are Unitaries* (~35 min)
**Goal:** the whole formalism in linear-algebra terms; entanglement as non-factorizability.
**Builds on:** [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb). &nbsp; **Feeds into:** Session 2 (the QFT).

---

## 2. No Mysticism Required

💡 **Intuition.** One qubit: a unit vector in $\mathbb{C}^2$. $n$ qubits: a unit vector in $\mathbb{C}^{2^n}$ — the exponential size of that space is the entire hardware story. Gates are [unitary matrices](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) (reversible, norm-preserving — Parseval's cousins); measurement samples index $k$ with probability $|\psi_k|^2$ and is the *only* nonlinear thing in the theory. **Entanglement** is simply a joint state that doesn't factor as a tensor product — correlation with no classical joint distribution behind it.

In [ ]:
# build the Bell state with H then CNOT; verify it cannot factor
# factorization test: a product state has rank-1 'amplitude matrix'

# YOUR CODE HERE


**What just happened.** The circuit produced amplitudes $(0.707, 0, 0, 0.707)$ — exactly $\frac{1}{\sqrt2}(|00\rangle+|11\rangle)$, so measurement gives 00 or 11 with 50/50 probability and 01/10 never happen. That's already suspicious-looking correlation, but the SVD is the proof: reshaping the length-4 amplitude vector into a $2\times2$ matrix and taking singular values gives **two** nonzero values ($0.707, 0.707$), not one. A product state $|\psi_1\rangle\otimes|\psi_2\rangle$ always reshapes to a rank-1 (one nonzero singular value) matrix — that's what a Kronecker product *is*, algebraically. Two nonzero singular values is a clean, checkable certificate that this state cannot be factored into two independent qubits: the correlation is baked into the amplitudes themselves, not layered on top of two separate stories.

---
### 🕐 Session 2 of 3 — *The QFT Is the FFT* (~40 min)
**Goal:** build the quantum Fourier transform from gates; verify it equals the DFT matrix exactly.
**Builds on:** Session 1; [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S7. &nbsp; **Feeds into:** Session 3 (what quantum actually speeds up).

---

## 3. Home Turf

💡 **Intuition.** The QFT on $n$ qubits applies the $2^n \times 2^n$ **DFT matrix** to the amplitude vector — built from $O(n^2)$ two-qubit gates, the same divide-and-conquer as the [radix-2 FFT](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) (the gate cascade IS the butterfly diagram). The catch every headline omits: the result lives in *amplitudes you cannot read out directly* — measuring gives one sample, not the spectrum. QFT speedups exist only where a global *property* of the spectrum (like a period) suffices — which is exactly what Shor's algorithm extracts.

In [ ]:
        # H on qubit j
        # controlled phases from qubits j+1..n
    # bit-reversal permutation (same one as the FFT!)

# YOUR CODE HERE


**What just happened.** Both checks bottom out at machine precision: `3.78e-15` against the hand-built DFT matrix, `1.49e-16` against `np.fft`. Those aren't "close" — for double-precision arithmetic accumulating $O(n^2)$ complex multiplications, error at the $10^{-15}$ level *is* zero; it's the same size as the rounding noise you'd get squaring and re-summing anything this many times. The gate cascade (Hadamards, controlled-phase rotations, then bit-reversal) computes the exact $2^n\times2^n$ DFT matrix — the same matrix Foundations 1 builds from butterflies, the same permutation every radix-2 FFT applies at the end. Nothing here is "quantum-flavored Fourier"; it's the DFT, factored into $O(n^2)$ two-qubit gates instead of computed by $O(n\log n)$ classical butterfly operations — and, per the discussion above, hidden inside a state you can only sample once.

In [ ]:
# period finding — the heart of Shor — on a simulated register

# YOUR CODE HERE


**What just happened.** The input state has amplitude only every $r=8$ positions out of $N=64$ — a period-8 "comb" in the register basis. After the QFT, measurement lands *only* at multiples of $N/r = 8$ (the printed outcomes: $0,8,16,\dots,56$), and every other position has exactly zero probability. This is the DSP fact in a quantum costume: the Fourier transform of a period-$r$ comb is itself a period-$(N/r)$ comb — sampling a periodic signal at its period concentrates all spectral energy at harmonics of the fundamental. **The catch this idealized demo hides:** it prepared the periodic state directly and used $r=8$, a power of two that divides $N=64$ exactly, so every peak lands exactly on an integer and the period is readable by eye. Real period-finding (as in Shor's algorithm) first has to *build* this periodic state from a modular-exponentiation oracle, and $r$ generally does *not* divide $N$ evenly — the peaks smear across neighboring bins, and recovering $r$ requires a continued-fractions post-processing step on the noisy measured value. This cell shows the clean endpoint of the idea, not the full algorithm.

---
### 🕐 Session 3 of 3 — *What Quantum Actually Speeds Up* (~30 min)
**Goal:** the honest scoreboard: where proofs exist, where hype lives, and what to watch.
**Builds on:** Session 2.

---

## 4. The Scoreboard

| Problem | Speedup | Status |
|---|---|---|
| Factoring / discrete log (Shor) | exponential | proven; needs ~millions of good qubits |
| Unstructured search (Grover) | quadratic only | proven; modest in practice |
| Simulating quantum systems | exponential | the original killer app — chemistry/materials |
| Generic ML / optimization | — | **no proven advantage**; data loading often eats the win |

💡 **Intuition.** The honest summary for an engineer: quantum computers are *interference machines* — they win when a problem's answer can be encoded so wrong paths cancel ([the QFT's](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) specialty) — and today's hardware fights decoherence with error-correction overheads of ~1000 physical per logical qubit. Track logical-qubit counts, not press releases. Your DSP training transfers verbatim: unitaries, interference, transforms — you already speak the language.

---
## Where next

- [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S7 — the butterfly you just rebuilt from gates.
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) — the entire formalism, secretly.